# Data Cleaning and Visualization – Titanic Dataset

**Objective:** Clean the Titanic passenger dataset, explore its structure, handle missing values and duplicates, inspect outliers, and create visualizations.

**Tools:** Python, Pandas, NumPy, Matplotlib, Jupyter Notebook.

> Run the cells from top to bottom. This notebook expects `titanic.csv` in the project's `data/` folder. If it is not found, it will try to download the public dataset.

## 1. Import libraries and set up project paths

In [ ]:
from pathlib import Path
import urllib.request

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Find the project root whether this notebook is run from notebooks/ or the root folder.
cwd = Path.cwd()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("Project folder:", PROJECT_ROOT)
print("Data folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)

## 2. Load the dataset

In [ ]:
data_path = DATA_DIR / "titanic.csv"

if not data_path.exists():
    dataset_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    print("Local dataset not found. Downloading Titanic dataset...")
    urllib.request.urlretrieve(dataset_url, data_path)

df = pd.read_csv(data_path)
print("Dataset loaded successfully.")
display(df.head())

## 3. Understand the data

In [ ]:
print("Dataset shape (rows, columns):", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nNumerical summary:")
display(df.describe())

## 4. Check missing values and duplicates

In [ ]:
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_summary = pd.DataFrame({
    "Missing Values": missing_values,
    "Missing Percentage": (missing_values / len(df) * 100).round(2)
})
display(missing_summary)

print("Duplicate rows:", df.duplicated().sum())

## 5. Clean the dataset

- Fill missing `Age` values with the median.
- Fill missing `Embarked` values with the mode.
- Fill missing `Cabin` values with the label `Unknown`.
- Remove duplicate rows.

The median is used for `Age` because it is less affected by extreme values than the mean. `Cabin` has many missing values, so missing entries are kept as an explicit category.

In [ ]:
cleaned_df = df.copy()

cleaned_df["Age"] = cleaned_df["Age"].fillna(cleaned_df["Age"].median())
cleaned_df["Embarked"] = cleaned_df["Embarked"].fillna(cleaned_df["Embarked"].mode()[0])
cleaned_df["Cabin"] = cleaned_df["Cabin"].fillna("Unknown")

rows_before = len(cleaned_df)
cleaned_df = cleaned_df.drop_duplicates().copy()
duplicates_removed = rows_before - len(cleaned_df)

print("Duplicate rows removed:", duplicates_removed)
print("\nMissing values after cleaning:")
display(cleaned_df.isnull().sum().to_frame("Missing Values"))

## 6. Detect outliers using the IQR method

The Interquartile Range (IQR) method is used to flag unusually high or low `Fare` values. Outliers are identified for inspection; they are not automatically deleted because unusually high fares may be genuine passenger records.

In [ ]:
Q1 = cleaned_df["Fare"].quantile(0.25)
Q3 = cleaned_df["Fare"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

fare_outliers = cleaned_df[
    (cleaned_df["Fare"] < lower_bound) |
    (cleaned_df["Fare"] > upper_bound)
]

print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")
print("Fare outliers detected:", len(fare_outliers))

## 7. Visualize the data

### 7.1 Passenger count by gender

In [ ]:
gender_counts = cleaned_df["Sex"].value_counts()

plt.figure(figsize=(6, 4))
gender_counts.plot(kind="bar")
plt.title("Passenger Count by Gender")
plt.xlabel("Gender")
plt.ylabel("Number of Passengers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 7.2 Age distribution

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(cleaned_df["Age"], bins=20, edgecolor="black")
plt.title("Age Distribution of Passengers")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### 7.3 Survival count

In [ ]:
survival_labels = cleaned_df["Survived"].map({0: "Did not survive", 1: "Survived"})
survival_counts = survival_labels.value_counts()

plt.figure(figsize=(6, 4))
survival_counts.plot(kind="bar")
plt.title("Passenger Survival Count")
plt.xlabel("Survival Status")
plt.ylabel("Number of Passengers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 7.4 Fare distribution

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(cleaned_df["Fare"], bins=30, edgecolor="black")
plt.title("Fare Distribution")
plt.xlabel("Fare")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### 7.5 Fare outlier box plot

In [ ]:
plt.figure(figsize=(7, 4))
plt.boxplot(cleaned_df["Fare"], vert=False)
plt.title("Box Plot of Passenger Fares")
plt.xlabel("Fare")
plt.tight_layout()
plt.show()

## 8. Save the cleaned dataset

In [ ]:
cleaned_file = OUTPUT_DIR / "cleaned_titanic.csv"
cleaned_df.to_csv(cleaned_file, index=False)

print("Cleaned dataset saved to:", cleaned_file)
print("Cleaned dataset shape:", cleaned_df.shape)

## 9. Summary

In this project, the Titanic dataset was loaded and explored using Pandas. Missing values in selected columns were handled, duplicate rows were removed, and potential fare outliers were identified using the IQR method. Matplotlib charts were created to explore passenger gender, age, survival, and fare distributions.

**Output:** `output/cleaned_titanic.csv`

### Note
This is an exploratory data cleaning and visualization project. It does not train a machine-learning model or make predictions.